### Import Dependencies

In [ ]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

### Download all data from Qdrant

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False,
)

In [ ]:
all_points[0][0].payload

In [ ]:
all_points[0]

In [ ]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]]

In [ ]:
all_context

### Render a prompt to generate synthetic Eval reference dataset


In [ ]:
output_schema = {
    "type": "object",
    "properties": {
        "items": {
            "type": "array",
            "description": "Synthetic eval examples (questions and reference chunk IDs).",
            "items": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "Suggested question.",
                    },
                    "chunk_ids": {
                        "type": "array",
                        "items": {
                            "type": "string",
                            "description": "ID of the chunk that could be used to answer the question.",
                        },
                    },
                    "answer_example": {
                        "type": "string",
                        "description": "Suggested answer grounded in the context.",
                    },
                    "reasoning": {
                        "type": "string",
                        "description": "Reasoning why the question could be answered with the chunks.",
                    },
                },
            },
        },
    },
    "required": ["items"],
}


SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

Your reply must be JSON only (no markdown fences). The root value must be an object with a single key "items" whose value is the array of example objects — not a bare array at the top level.

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""


In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
print(USER_PROMPT)

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

In [ ]:
parsed = json.loads(response.choices[0].message.content)
json_output = parsed["items"] if isinstance(parsed, dict) and "items" in parsed else parsed


In [ ]:
json_output

In [ ]:
len(json_output)

In [ ]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

In [ ]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

In [ ]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0C4NJPN4Q")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [ ]:
points[0].payload

In [ ]:
def get_description(parent_asin: str) -> str:

    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["description"]


In [ ]:
get_description("B0C4NJPN4Q")

### Create Eval dataset in LangSmith

In [ ]:
ls_client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [ ]:
dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset"
)

In [ ]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"],
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(chunk_id) for chunk_id in item["chunk_ids"]]
        }
    )